# 🏆 17. Modelos benchmark: mejores combinaciones y artefactos

La tercera entrega se concentra en la tarea de **regresión** (venta del día siguiente). De las 28 combinaciones de regresión de la entrega 2 (7 modelos × 4 optimizadores), el tablero no incluye todas, sino **la mejor de cada modelo**, elegida con un criterio estadístico y no por apreciación visual:

1. Para cada uno de los 7 modelos se elige el optimizador con **menor RMSE log medio en el bucle externo** de la validación anidada.
2. Se verifica que la **mejor combinación global** pertenezca al conjunto de finalistas de la comparación estadística (**MCS** al 90 %).
3. Se agrega **HistGradientBoosting** (boosting por histogramas, la implementación de scikit-learn inspirada en LightGBM; Ke et al., 2017) como referencia del estado del arte, con los mismos 4 optimizadores y la misma validación anidada.

In [1]:
import json
import sys
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
from comun import estilo, SEED
from datos import partir, preparar, X_COLS
from experimentos import armar_maestra, cargar, fila_maestra

estilo()
pd.set_option("display.max_colwidth", 80)
train, test = partir(preparar())
maestra = armar_maestra()
reg = maestra[maestra["tarea"] == "reg"]
OPT = {"grid": "Grid", "random": "Random", "bayes": "Bayes", "genetico": "Genético"}

/opt/miniconda3/envs/ml_venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# selección: mejor optimizador de cada modelo (menor RMSE log medio del bucle externo)
hgb = pd.DataFrame([fila_maestra(cargar("reg", "hgb", "ninguno", o)) for o in OPT])
todas = pd.concat([reg, hgb], ignore_index=True)
mejores = todas.loc[todas.groupby("modelo")["rmse_log_media"].idxmin()].sort_values("rmse_log_media")
mcs = pd.read_csv("resultados/original_mcs_rmse.csv", index_col=0)
NOMBRE = {"knn": "KNN", "ridge": "Ridge", "lasso": "Lasso", "arbol": "Árbol", "rf": "Random Forest", "xgb": "XGBoost",
          "svr": "SVR lineal", "hgb": "HistGradientBoosting"}
sel = pd.DataFrame({
    "modelo": mejores["modelo"].map(NOMBRE), "optimizador elegido": mejores["optimizador"].map(OPT),
    "RMSE log (media ± desv.)": [f"{a:.4f} ± {b:.4f}" for a, b in zip(mejores["rmse_log_media"], mejores["rmse_log_desv"])],
    "MAE log (media ± desv.)": [f"{a:.4f} ± {b:.4f}" for a, b in zip(mejores["mae_log_media"], mejores["mae_log_desv"])],
    "2.º optimizador (RMSE log)": [f"{todas[todas['modelo'] == m].nsmallest(2, 'rmse_log_media')['rmse_log_media'].iloc[1]:.4f}" for m in mejores["modelo"]],
    "en el MCS (90 %)": [bool(mcs.loc[f"{NOMBRE[m]} · {OPT[o]}", "en el conjunto"]) for m, o in zip(mejores["modelo"], mejores["optimizador"])],
    "motivo": ["menor RMSE log medio de sus 4 optimizadores"] * len(mejores)})
sel.iloc[0, -1] = "mejor combinación global; p MCS = 1,00"
sel.set_index("modelo")

,optimizador elegido,RMSE log (media ± desv.),MAE log (media ± desv.),2.º optimizador (RMSE log),en el MCS (90 %),motivo
modelo,,,,,,
Random Forest,Grid,0.8874 ± 0.1266,0.2906 ± 0.0505,0.8923,True,"mejor combinación global; p MCS = 1,00"
XGBoost,Random,0.8944 ± 0.1222,0.2961 ± 0.0479,0.8977,True,menor RMSE log medio de sus 4 optimizadores
HistGradientBoosting,Random,0.9047 ± 0.1346,0.3071 ± 0.0436,0.9064,True,menor RMSE log medio de sus 4 optimizadores
KNN,Genético,0.9314 ± 0.1112,0.3050 ± 0.0489,0.9370,False,menor RMSE log medio de sus 4 optimizadores
Lasso,Bayes,0.9493 ± 0.0894,0.3040 ± 0.0368,0.9493,False,menor RMSE log medio de sus 4 optimizadores
Ridge,Genético,0.9498 ± 0.0889,0.3038 ± 0.0352,0.9504,False,menor RMSE log medio de sus 4 optimizadores
Árbol,Bayes,0.9516 ± 0.1143,0.3324 ± 0.0414,0.9587,False,menor RMSE log medio de sus 4 optimizadores
SVR lineal,Grid,0.9883 ± 0.0963,0.3083 ± 0.0272,0.9886,False,menor RMSE log medio de sus 4 optimizadores


**Lectura.** La mejor combinación global es **Random Forest · Grid** (RMSE log 0,887 ± 0,127), y está en el conjunto de confianza del MCS con p = 1,00: es el modelo de referencia contra el que se compara el modelo original (cap. 20). XGBoost y HistGradientBoosting también quedan en el conjunto; KNN, los lineales, el árbol y el SVR no. Para los modelos de 1 o 2 hiperparámetros (Ridge, Lasso, SVR) la diferencia con el segundo optimizador es de milésimas: la elección del optimizador casi no importa (cap. 10). **HistGradientBoosting**, el método más reciente, queda 3.º, detrás de Random Forest y XGBoost: en este problema el estado del arte del boosting por histogramas no aporta sobre los dos ensambles clásicos.

## 17.1 Reentrenamiento final y serialización

Con los hiperparámetros óptimos (elegidos por la búsqueda sobre todo train, la misma de la entrega 2), cada pipeline completo (imputación, escalado, one-hot, modelo y transformación log del objetivo) se reentrena con **todo train** y se guarda con `joblib.dump(..., compress=3)` en `artifacts/`. Junto a cada modelo se guarda un `metadata.json` con las versiones de las librerías, la semilla, los hiperparámetros finales, el esquema de entrada (nombre, tipo y rango válido de cada columna) y las métricas del bucle externo. El código está en `book/artefactos.py` y `book/exportar_app.py`.

In [3]:
ART = Path.cwd().parent / "artifacts"
filas = []
for f in sorted(ART.glob("*.metadata.json")):
    m = json.loads(f.read_text())
    filas.append({"artefacto": f.name.replace(".metadata.json", ".joblib"), "MB": m.get("mb_en_disco"),
                  "optimizador": m.get("optimizador"), "semilla": m.get("semilla"),
                  "hiperparámetros": json.dumps(m["hiperparametros"], default=float)[:80],
                  "ida y vuelta": m.get("prueba_ida_y_vuelta"), "columnas de entrada": len(m["esquema_entrada"])})
pd.DataFrame(filas).set_index("artefacto").round(2)

,MB,optimizador,semilla,hiperparámetros,ida y vuelta,columnas de entrada
artefacto,,,,,,
acs.joblib,22.16,"bayes (TPE, 16 evaluaciones)",42,"{""arbol"": {""max_depth"": 2, ""min_samples_leaf"": 318, ""lam"": 0.013554586987236...",True,14
arbol.joblib,0.04,bayes,42,"{""max_depth"": 16, ""min_samples_leaf"": 23}",True,14
hgb.joblib,0.09,random,42,"{""max_iter"": 178, ""learning_rate"": 0.04891065903176344, ""max_leaf_nodes"": 16...",True,14
knn.joblib,1.70,genetico,42,"{""n_neighbors"": 12, ""weights"": ""distance""}",True,14
lasso.joblib,0.00,bayes,42,"{""alpha"": 0.0024810409748678114}",True,14
rf.joblib,17.78,grid,42,"{""max_depth"": 16, ""max_features"": 0.6, ""min_samples_leaf"": 2, ""n_estimators""...",True,14
ridge.joblib,0.00,genetico,42,"{""alpha"": 26.231751227258428}",True,14
svr.joblib,0.00,grid,42,"{""C"": 0.0071968567300115215, ""epsilon"": 0.0}",True,14
xgb.joblib,0.14,random,42,"{""n_estimators"": 373, ""learning_rate"": 0.016085381438354866, ""max_depth"": 4,...",True,14


In [4]:
# un metadata.json completo, como ejemplo
print(json.dumps(json.loads((ART / "rf.metadata.json").read_text()), indent=2, ensure_ascii=False)[:2500])

{
  "modelo": "rf",
  "optimizador": "grid",
  "semilla": 42,
  "versiones": {
    "python": "3.9.25",
    "numpy": "1.26.4",
    "pandas": "2.1.0",
    "scikit-learn": "1.3.0",
    "xgboost": "2.1.4",
    "joblib": "1.3.2"
  },
  "hiperparametros": {
    "max_depth": 16,
    "max_features": 0.6,
    "min_samples_leaf": 2,
    "n_estimators": 150
  },
  "objetivo": "venta_total (COP), entrenado en log1p",
  "entrenado_con": "2026-07-15 a 2026-09-02 (22205 filas)",
  "esquema_entrada": [
    {
      "nombre": "lag1",
      "tipo": "real",
      "min": 0.0,
      "max": 19.451178575072152,
      "admite_nulos": false
    },
    {
      "nombre": "lag7",
      "tipo": "real",
      "min": 0.0,
      "max": 19.451178575072152,
      "admite_nulos": false
    },
    {
      "nombre": "lag14",
      "tipo": "real",
      "min": 0.0,
      "max": 19.451178575072152,
      "admite_nulos": false
    },
    {
      "nombre": "media7",
      "tipo": "real",
      "min": 0.0,
      "max": 19.20312

## 17.2 Prueba de ida y vuelta (*round-trip*)

Se carga cada artefacto desde disco y se compara su predicción sobre el test fijo (3 – 23 sep) con la de un modelo reentrenado en memoria con los mismos hiperparámetros y la misma semilla. Deben ser **idénticas**. La misma prueba se repitió en un entorno limpio con **Python 3.11.9** y las versiones de `requirements.txt` (el entorno de Render): la diferencia máxima fue ≤ 5·10⁻⁶ COP en los 9 modelos.

In [5]:
from modelos import ajustar, crear_modelo
filas = []
for m, o in zip(mejores["modelo"], mejores["optimizador"]):
    params = cargar("reg", m, "ninguno", o)["params_final"]
    mem = crear_modelo("reg", m, params, semilla=SEED)
    ajustar(mem, train[X_COLS], train["venta_total"].to_numpy(), "reg", m, "ninguno")
    t = time.perf_counter(); disco = joblib.load(ART / f"{m}.joblib"); t_carga = time.perf_counter() - t
    p_mem, p_disco = mem.predict(test[X_COLS]), disco.predict(test[X_COLS])
    filas.append({"modelo": NOMBRE[m], "predicciones idénticas": bool(np.array_equal(p_mem, p_disco)),
                  "máx |diferencia| (COP)": float(np.max(np.abs(p_mem - p_disco))), "seg de carga": t_carga})
pd.DataFrame(filas).set_index("modelo")

,predicciones idénticas,máx |diferencia| (COP),seg de carga
modelo,,,
Random Forest,True,0.00,0.12
XGBoost,True,0.00,0.01
HistGradientBoosting,True,0.00,0.00
KNN,True,0.00,0.01
Lasso,True,0.00,0.00
Ridge,True,0.00,0.00
Árbol,True,0.00,0.00
SVR lineal,True,0.00,0.00


**Tamaño de los artefactos.** El más pesado es Random Forest (17,8 MB comprimido; 150 árboles de profundidad 16) y el ACS (20,8 MB, porque incluye a sus cuatro expertos). Todos están por debajo del límite de 100 MB por archivo de GitHub, así que no hizo falta Git LFS. En el servidor, el tablero carga los modelos **bajo demanda** y mantiene a lo sumo dos en memoria, para no superar el límite de 512 MB del plan gratuito de Render (cap. 21).

## 17.3 Tabla maestra final

Las 140 combinaciones de la entrega 2, los 4 HistGradientBoosting y el modelo original, con las métricas del bucle externo (media y desviación), las de test, los tiempos y la semilla.

In [6]:
import pickle
from comun import DATA, a_log
r = pickle.load(open(DATA / "processed" / "modelo_original.pkl", "rb"))
res = pd.read_csv("resultados/original_resumen.csv", index_col=0).loc["ACS (original)"]
yt = test["venta_total"].to_numpy()
acs = {"tarea": "reg", "modelo": "acs (original)", "balanceo": "ninguno", "optimizador": "bayes", "semilla": SEED,
       "n_evals": 16 * 5, "rmse_log_media": res["RMSE log media"], "rmse_log_desv": res["RMSE log desv"],
       "mae_log_media": res["MAE log media"], "mae_log_desv": res["MAE log desv"],
       "rmse_log_test": float(np.sqrt(np.mean((a_log(yt) - a_log(r["pred_test"])) ** 2))),
       "mae_log_test": float(np.mean(np.abs(a_log(yt) - a_log(r["pred_test"])))),
       "seg_fit_final": r["seg_fit_final"], "seg_pred_test": r["seg_pred_test"],
       "params_final": json.dumps(r["params_final"], default=float)}
final = pd.concat([maestra, hgb, pd.DataFrame([acs])], ignore_index=True)
final.to_csv("resultados/maestra_final.csv", index=False)
print(f"{len(final)} filas: {final['tarea'].value_counts().to_dict()}")
final[final["tarea"] == "reg"].nsmallest(8, "rmse_log_media")[
    ["modelo", "optimizador", "rmse_log_media", "rmse_log_desv", "mae_log_media", "rmse_log_test", "seg_fit_final", "semilla"]].style.format(precision=4)

145 filas: {'clf': 112, 'reg': 33}


,modelo,optimizador,rmse_log_media,rmse_log_desv,mae_log_media,rmse_log_test,seg_fit_final,semilla
16,rf,grid,0.8874,0.1266,0.2906,0.6221,11.3261,42
17,rf,random,0.8923,0.1298,0.2939,0.6237,3.2444,42
21,xgb,random,0.8944,0.1222,0.2961,0.6374,0.5982,42
18,rf,bayes,0.8945,0.1168,0.2932,0.6277,9.3661,42
23,xgb,genetico,0.8977,0.1324,0.2951,0.6304,0.9696,42
144,acs (original),bayes,0.8981,0.1268,0.2878,0.6108,71.8749,42
22,xgb,bayes,0.8981,0.1384,0.2930,0.6334,0.7721,42
19,rf,genetico,0.8998,0.1171,0.2939,0.6279,8.1663,42
